# Experiment: SpectralBio Final Accept Hardening Part 5 (A100)

This notebook is the protocol-robustness notebook.

## Scope
- Sweep local window radii 20, 40, 80, and 120
- Sweep layer protocols last-4, last-8, top-half, and all-layers
- Compare fixed alpha, tuned alpha, and ESM-1v-augmented alpha
- Run across TP53, BRCA1, BRCA2, and NSD1 with 150M, 650M, and selected 3B paths

## Intended runtime
- Target hardware: A100
- This notebook turns the scaling criticism into a protocol-robustness result


In [ ]:
from pathlib import Path

USE_GOOGLE_DRIVE = False
DRIVE_MOUNT_POINT = Path('/content/drive')
DRIVE_OUTPUT_SUBDIR = Path('MyDrive/SpectralBioFinalAcceptA100')

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount(str(DRIVE_MOUNT_POINT))
    print('Drive mounted at', DRIVE_MOUNT_POINT)
else:
    print('Drive mount skipped; outputs stay in the VM unless OUTPUT_ROOT is changed later.')


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/DaviBonetto/SpectralBio.git'
REPO_BRANCH = 'codex/claw4s-rebuild'
DEFAULT_REPO_ROOT = Path('/content/Stanford-Claw4s')
ENV_REPO_ROOT = os.environ.get('SPECTRALBIO_REPO_ROOT', '').strip()


def _looks_like_repo(path: Path) -> bool:
    return (path / 'src' / 'spectralbio').exists() and (path / 'notebooks').exists()


candidate_roots = []
if ENV_REPO_ROOT:
    candidate_roots.append(Path(ENV_REPO_ROOT).expanduser())
candidate_roots.extend([Path.cwd(), Path.cwd().parent, DEFAULT_REPO_ROOT])

REPO_ROOT = next((path.resolve() for path in candidate_roots if _looks_like_repo(path)), DEFAULT_REPO_ROOT)
BOOTSTRAP_MARKER = REPO_ROOT / '.colab_bootstrap_v3_complete'

if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])

os.chdir(REPO_ROOT)
subprocess.check_call(['git', 'fetch', 'origin', REPO_BRANCH])
subprocess.check_call(['git', 'checkout', REPO_BRANCH])
src_path = REPO_ROOT / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

if not BOOTSTRAP_MARKER.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install',
        'numpy==2.1.3', 'plotly==5.24.1', 'pyyaml==6.0.2', 'scikit-learn==1.5.2',
        'scipy==1.14.1', 'transformers==4.49.0', 'pandas', 'matplotlib'
    ])
    BOOTSTRAP_MARKER.write_text('ok\n', encoding='utf-8')
    print('Dependencies installed. Continuing in the same runtime.')
else:
    print('Bootstrap marker found; skipping reinstall.')



## Plan

- Stop at a compact, predeclared sweep rather than an unbounded hyperparameter search.
- Cover the reviewer's obvious escape routes: windowing, layer choice, checkpoint choice, and combination rule.
- Keep TP53, BRCA2, BRCA1, and NSD1 fixed across the grid for comparability.


In [ ]:
import subprocess
import sys
import pandas as pd

try:
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_protocol_sweep_suite,
    )
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.', '--no-deps'])
    from spectralbio.supplementary.final_accept_hardening import (
        AcceptHardeningConfig,
        create_accept_hardening_paths,
        run_protocol_sweep_suite,
    )

GENES = ('TP53', 'BRCA1', 'BRCA2', 'NSD1')
WINDOW_RADII = (20, 40, 80, 120)
LAYER_PROTOCOLS = ('last4', 'last8', 'top_half', 'all_layers')
ALPHA_PROTOCOLS = ('fixed_055', 'nested_best', 'esm1v_augmented')
MODELS = ('facebook/esm2_t30_150M_UR50D', 'facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D')
OVERWRITE = False

OUTPUT_ROOT = (
    DRIVE_MOUNT_POINT / DRIVE_OUTPUT_SUBDIR
    if USE_GOOGLE_DRIVE
    else REPO_ROOT / 'supplementary' / 'final_accept_a100'
)

config = AcceptHardeningConfig(
    stronger_model_names=('facebook/esm2_t33_650M_UR50D', 'facebook/esm2_t36_3B_UR50D'),
    overwrite=OVERWRITE,
)
paths = create_accept_hardening_paths(repo_root=REPO_ROOT, output_root=OUTPUT_ROOT)
print(paths)
print('Genes:', GENES)
print('Window radii:', WINDOW_RADII)
print('Layer protocols:', LAYER_PROTOCOLS)



In [ ]:
protocol_summary = run_protocol_sweep_suite(
    paths=paths,
    config=config,
    genes=GENES,
    window_radii=WINDOW_RADII,
    layer_protocols=LAYER_PROTOCOLS,
    alpha_protocols=ALPHA_PROTOCOLS,
    model_names=MODELS,
)
display(pd.DataFrame(protocol_summary['rows']).head())



In [ ]:
expected_paths = [
    paths.root / 'protocol_sweep' / 'protocol_sweep_long.csv',
    paths.root / 'protocol_sweep' / 'protocol_sweep_summary.json',
    paths.root / 'protocol_sweep' / 'protocol_sweep_tp53_focus.csv',
]
for path in expected_paths:
    print(path)


In [ ]:
print('Part 5 scaffold complete. This notebook is reserved for the A100 protocol sweep.')
